# Remover20 — remove backgrounds with Replicate bria/remove-background

This notebook removes backgrounds from images in `Photos/` using the Replicate model `bria/remove-background` and writes PNGs with transparency into `photos_wo_background/`.

- Input: `Photos/`
- Output: `photos_wo_background/`
- Token: set `REPLICATE_API_TOKEN` (cell below)



In [19]:
import os

# Set Replicate API token for this notebook session (replace if needed)
os.environ["REPLICATE_API_TOKEN"] = "YOUR_REPLICATE_API_TOKEN_HERE"
print("REPLICATE_API_TOKEN set for this session.")


REPLICATE_API_TOKEN set for this session.


In [20]:
# Optional: install runtime deps (uncomment if needed)
# !pip install --quiet replicate pillow requests tqdm

import os
import io
from pathlib import Path
from typing import Optional

import requests
from PIL import Image
from tqdm import tqdm

try:
    import replicate
except ImportError as e:
    raise ImportError("replicate package is required. Install with `pip install replicate`. ") from e

WORKSPACE = "/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator"
INPUT_DIR = os.path.join(WORKSPACE, "Photos")
OUTPUT_DIR = os.path.join(WORKSPACE, "photos_wo_background")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_SLUG = "bria/remove-background"
print("Using model:", MODEL_SLUG)

if not os.environ.get("REPLICATE_API_TOKEN"):
    print("WARNING: REPLICATE_API_TOKEN is not set. Set it via os.environ or shell env before running.")


Using model: bria/remove-background


In [21]:
def remove_background_with_replicate(input_image_path: str) -> Image.Image:
    """Run bria/remove-background on Replicate and return an RGBA PIL image."""
    with open(input_image_path, "rb") as f:
        # bria/remove-background returns a File object-like result with .url() and bytes interface
        output = replicate.run(MODEL_SLUG, input={"image": f})

    # Try official File-like behaviors
    # - Some SDKs return an object with .url() and can be written as bytes
    # - In Python, it may be a dict or a URL string; handle robustly
    # URL first
    result_url = None
    if hasattr(output, "url") and callable(getattr(output, "url")):
        result_url = output.url()
    elif isinstance(output, str):
        result_url = output
    elif isinstance(output, (list, tuple)) and output and isinstance(output[0], str):
        result_url = output[0]
    elif isinstance(output, dict):
        for key in ("url", "output", "image", "result"):
            val = output.get(key)
            if isinstance(val, str):
                result_url = val
                break
            if isinstance(val, (list, tuple)) and val and isinstance(val[0], str):
                result_url = val[0]
                break

    if result_url:
        resp = requests.get(result_url, timeout=60)
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content)).convert("RGBA")

    # Fallback: if output is bytes-like
    if isinstance(output, (bytes, bytearray)):
        return Image.open(io.BytesIO(output)).convert("RGBA")

    raise RuntimeError(f"Unexpected output from Replicate: {type(output)} -> {output}")


In [22]:
def build_output_path(input_path: str, output_dir: str) -> str:
    name = Path(input_path).stem
    return str(Path(output_dir) / f"{name}.png")


def process_folder(input_dir: str = INPUT_DIR, output_dir: str = OUTPUT_DIR) -> None:
    supported = {".jpg", ".jpeg", ".png"}
    image_paths = [
        os.path.join(input_dir, name)
        for name in os.listdir(input_dir)
        if os.path.splitext(name)[1].lower() in supported
    ]

    if not image_paths:
        print("No images found in:", input_dir)
        return

    for path in tqdm(image_paths, desc="Removing backgrounds"):
        try:
            out_path = build_output_path(path, output_dir)
            if os.path.exists(out_path):
                # Skip already processed
                continue
            rgba = remove_background_with_replicate(path)
            Path(out_path).parent.mkdir(parents=True, exist_ok=True)
            rgba.save(out_path, format="PNG")
        except Exception as e:
            print(f"Failed: {path} -> {e}")

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print("Ready. Run process_folder() to start.")


Input:  /Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/Photos
Output: /Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/photos_wo_background
Ready. Run process_folder() to start.


In [23]:
# Override helper to handle Replicate FileOutput, lists, dicts, URLs, and bytes
import io
import requests
from PIL import Image


def _extract_bytes_from_replicate_output(output) -> bytes:
    # Single file-like with read()
    if hasattr(output, "read") and callable(getattr(output, "read")):
        data = output.read()
        if isinstance(data, (bytes, bytearray)):
            return bytes(data)

    # Has url property or method
    if hasattr(output, "url"):
        url_attr = getattr(output, "url")
        url_val = url_attr() if callable(url_attr) else url_attr
        if isinstance(url_val, str) and url_val.startswith("http"):
            resp = requests.get(url_val, timeout=60)
            resp.raise_for_status()
            return resp.content

    # List/tuple: take first valid
    if isinstance(output, (list, tuple)) and output:
        return _extract_bytes_from_replicate_output(output[0])

    # Dict: common keys
    if isinstance(output, dict):
        for key in ("url", "output", "image", "result"):
            if key in output:
                return _extract_bytes_from_replicate_output(output[key])

    # Raw URL string
    if isinstance(output, str) and output.startswith("http"):
        resp = requests.get(output, timeout=60)
        resp.raise_for_status()
        return resp.content

    # Raw bytes
    if isinstance(output, (bytes, bytearray)):
        return bytes(output)

    raise RuntimeError(f"Unexpected output from Replicate: {type(output)} -> {output}")


def remove_background_with_replicate(input_image_path: str) -> Image.Image:
    with open(input_image_path, "rb") as f:
        output = replicate.run(MODEL_SLUG, input={"image": f})
    image_bytes = _extract_bytes_from_replicate_output(output)
    return Image.open(io.BytesIO(image_bytes)).convert("RGBA")



In [24]:
process_folder()

Removing backgrounds: 100%|█████████████████████| 66/66 [12:16<00:00, 11.15s/it]
